## Gather models performances

From each trained model in *results/BERT_models/NACE_classification* take 

- results.json
- training_config.json

and compare in one list

In [1]:
import glob
import json 
import os
import pandas as pd

os.chdir("projects/nace_classification/nace_report_topic_analysis/")

In [2]:
%pwd

'/data/resources/weichel-llama3/work/projects/nace_classification/nace_report_topic_analysis'

In [3]:
# retrieve all trained models path
BERT_models_path = "results/BERT_models/NACE_classification/"
trained_models_paths = glob.glob(os.path.join(BERT_models_path, "*"))
trained_models_paths.extend(glob.glob(os.path.join(BERT_models_path, "*", "*")))

# get only last level
trained_models_paths = [path for path in trained_models_paths if "_results__" in os.path.basename(path)]

In [4]:
path = trained_models_paths[2]

In [5]:
path

'results/BERT_models/NACE_classification/001__results__data_approach_1__desc_lvl_level_2__num_layers_2__cos_thres_0asdfa.4bert-base-uncased__train_full_model__some_labels__only_labels'

In [7]:
try: 
    with open(os.path.join(path, "training_config.json"), "r") as f: 
        training_config = json.load(f)

    with open(os.path.join(path, "results.json"), "r") as f: 
        results = json.load(f)
except FileNotFoundError: 
    pass

#training_config, results

In [8]:
def flatten_dict(d, parent_key="", sep="__"):
    """
    Recursively flattens a nested dict.
    {'a': 1, 'b': {'c': 2}} -> {'a': 1, 'b__c': 2}
    """
    items = {}
    for k, v in d.items():
        new_key = f"{parent_key}{sep}{k}" if parent_key else k
        if isinstance(v, dict):
            items.update(flatten_dict(v, new_key, sep=sep))
        else:
            items[new_key] = v
    return items

summary_rows = []
class_rows = []

In [9]:
for path in trained_models_paths: 
    # load
    try: 
        with open(os.path.join(path, "training_config.json"), "r") as f: 
            training_config = json.load(f)

        with open(os.path.join(path, "results.json"), "r") as f: 
            results = json.load(f)
    
    except FileNotFoundError: 
        continue

    run_name = os.path.basename(os.path.normpath(path))  # folder name for identification

    # --------- 1) RUN-LEVEL SUMMARY ROW ---------
    # flatten config (handles test_distribution etc.)
    flat_config = flatten_dict(training_config)

    # extract overall metrics from results.json (sklearn classification_report style)
    accuracy = results.get("accuracy", None)
    macro = results.get("macro avg", {})
    weighted = results.get("weighted avg", {})

    summary_row = {
        "run_name": run_name,
        "results__accuracy": accuracy,
        "results__macro_precision": macro.get("precision"),
        "results__macro_recall": macro.get("recall"),
        "results__macro_f1": macro.get("f1-score"),
        "results__weighted_precision": weighted.get("precision"),
        "results__weighted_recall": weighted.get("recall"),
        "results__weighted_f1": weighted.get("f1-score"),
    }

    # add flattened training_config fields
    summary_row.update(flat_config)

    summary_rows.append(summary_row)

    # --------- 2) PER-CLASS METRICS ROWS (optional but useful) ---------
    for label, metrics in results.items():
        # skip aggregate metrics
        if label in ["accuracy", "macro avg", "weighted avg"]:
            continue
        if not isinstance(metrics, dict):
            continue

        class_rows.append({
            "run_name": run_name,
            "class_id": label,  # '0', '1', ... from classification_report
            "precision": metrics.get("precision"),
            "recall": metrics.get("recall"),
            "f1": metrics.get("f1-score"),
            "support": metrics.get("support"),
        })

# finally build the DataFrames
df_runs = pd.DataFrame(summary_rows)
df_classes = pd.DataFrame(class_rows)

# ---- Add dataset version ----
get_ds_version = lambda x : os.path.basename(x).split("dataset__")[1].split("_sentence_len_")[0][-1]
df_runs["dataset_version"] = df_runs["data_path"].apply(get_ds_version)

# ---- Add data aggregation approach ----
get_data_agg_version = lambda x : x.split("training_data/")[1].split("approach_")[1][0] if len(x.split("training_data/")[1].split("approach_")) > 1 else None
df_runs["data_agg_approach"] = df_runs["data_path"].apply(get_data_agg_version)

# ---- Reorder columns: run_name | config_* | results_* ----
cols = df_runs.columns.tolist()

run_col = ["run_name"]
config_cols = [c for c in cols if c != "run_name" and not c.startswith("results__")]
result_cols = [c for c in cols if c.startswith("results__")]

df_runs = df_runs[run_col + config_cols + result_cols]
first_cols = ["run_name", "data_path", "data_agg_approach", "dataset_version"]
df_runs = df_runs[first_cols + [col for col in df_runs.columns if col not in first_cols]]

# (Optional) sort runs by macro F1 for quick comparison
df_runs = df_runs.sort_values("results__macro_f1", ascending=False)

In [10]:
df_runs

,run_name,data_path,data_agg_approach,dataset_version,train_full_model,all_labels,model_name,num_layers,new_thresh,only_labels,...,test_distribution__NO_CLASS,test_distribution__P,test_distribution__Q,results__accuracy,results__macro_precision,results__macro_recall,results__macro_f1,results__weighted_precision,results__weighted_recall,results__weighted_f1
32,037_results__data_approach_3__desc_lvl_level_1...,/data/resources/weichel-llama3/work/projects/n...,2,2,True,False,bert-base-uncased,1,0.50,True,...,NaN,47.0,17,0.965305,0.938520,0.957195,0.946704,0.966048,0.965305,0.965456
31,036_results__data_approach_3__desc_lvl_level_1...,/data/resources/weichel-llama3/work/projects/n...,2,2,True,False,bert-base-uncased,1,0.45,True,...,NaN,132.0,54,0.954133,0.924687,0.943708,0.933354,0.955300,0.954133,0.954459
7,018_results__data_approach_3__desc_lvl_level_1...,/data/resources/weichel-llama3/work/projects/n...,2,1,True,False,bert-base-uncased,1,0.45,True,...,NaN,75.0,63,0.951901,0.927193,0.938178,0.931847,0.953276,0.951901,0.952150
17,019_results__data_approach_3__desc_lvl_level_1...,/data/resources/weichel-llama3/work/projects/n...,2,1,True,False,bert-base-uncased,1,0.50,True,...,NaN,27.0,24,0.956652,0.915855,0.936532,0.925211,0.958209,0.956652,0.956972
36,041_results__data_approach_3__desc_lvl_level_1...,/data/resources/weichel-llama3/work/projects/n...,2,2,True,False,bert-base-uncased,2,0.45,True,...,NaN,132.0,54,0.948339,0.910982,0.933302,0.920768,0.950092,0.948339,0.948781
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,070_results__data_approach_3__desc_lvl_level_1...,/data/resources/weichel-llama3/work/projects/n...,3,2,True,False,bert-base-uncased,3,0.45,False,...,651.0,27.0,36,0.197815,0.114511,0.289964,0.137120,0.082443,0.197815,0.099736
60,069_results__data_approach_3__desc_lvl_level_1...,/data/resources/weichel-llama3/work/projects/n...,3,2,True,False,bert-base-uncased,3,0.50,True,...,NaN,5.0,8,0.320276,0.094470,0.171632,0.106078,0.143548,0.320276,0.184058
62,071_results__data_approach_3__desc_lvl_level_1...,/data/resources/weichel-llama3/work/projects/n...,3,2,True,False,bert-base-uncased,3,0.50,False,...,651.0,5.0,8,0.178802,0.072062,0.302345,0.102292,0.066013,0.178802,0.085749
57,066_results__data_approach_3__desc_lvl_level_1...,/data/resources/weichel-llama3/work/projects/n...,3,2,True,False,bert-base-uncased,2,0.50,False,...,651.0,5.0,8,0.191705,0.050726,0.254980,0.081151,0.040143,0.191705,0.065324


In [12]:
df_runs[~df_runs["only_labels"]]

,run_name,data_path,data_agg_approach,dataset_version,train_full_model,all_labels,model_name,num_layers,new_thresh,only_labels,...,test_distribution__NO_CLASS,test_distribution__P,test_distribution__Q,results__accuracy,results__macro_precision,results__macro_recall,results__macro_f1,results__weighted_precision,results__weighted_recall,results__weighted_f1
0,049_results__data_approach_2__desc_lvl_level_1...,/data/resources/weichel-llama3/work/projects/n...,2,2,True,False,bert-base-uncased,1,0.40,False,...,7248.0,340.0,154,0.843497,0.808437,0.898051,0.846778,0.847197,0.843497,0.837913
64,073_results__data_approach_2__desc_lvl_level_1...,/data/resources/weichel-llama3/work/projects/n...,2,2,True,False,bert-base-uncased,2,0.40,False,...,7248.0,340.0,154,0.834511,0.800561,0.893950,0.840008,0.839387,0.834511,0.827618
50,059_results__data_approach_2__desc_lvl_level_1...,/data/resources/weichel-llama3/work/projects/n...,2,2,True,False,bert-base-uncased,1,0.45,False,...,7248.0,132.0,54,0.846796,0.765795,0.911454,0.825137,0.867359,0.846796,0.846117
65,074_results__data_approach_2__desc_lvl_level_1...,/data/resources/weichel-llama3/work/projects/n...,2,2,True,False,bert-base-uncased,2,0.45,False,...,7248.0,132.0,54,0.839923,0.758507,0.906924,0.819469,0.860535,0.839923,0.839223
63,072_results__data_approach_2__desc_lvl_level_1...,/data/resources/weichel-llama3/work/projects/n...,2,2,True,False,bert-base-uncased,1,0.50,False,...,7248.0,47.0,17,0.894964,0.737652,0.912879,0.805864,0.911985,0.894964,0.898887
45,054_results__data_approach_3__desc_lvl_level_1...,/data/resources/weichel-llama3/work/projects/n...,3,2,True,False,bert-base-uncased,1,0.40,False,...,651.0,66.0,103,0.695347,0.687234,0.820066,0.711305,0.734390,0.695347,0.628306
66,075_results__data_approach_2__desc_lvl_level_1...,/data/resources/weichel-llama3/work/projects/n...,2,2,True,False,bert-base-uncased,2,0.50,False,...,7248.0,47.0,17,0.883921,0.657316,0.805462,0.706005,0.903451,0.883921,0.888667
52,061_results__data_approach_3__desc_lvl_level_1...,/data/resources/weichel-llama3/work/projects/n...,3,2,True,False,bert-base-uncased,2,0.40,False,...,651.0,66.0,103,0.651689,0.599365,0.786108,0.666467,0.545408,0.651689,0.584060
47,056_results__data_approach_3__desc_lvl_level_1...,/data/resources/weichel-llama3/work/projects/n...,3,2,True,False,bert-base-uncased,1,0.45,False,...,651.0,27.0,36,0.564117,0.586880,0.843141,0.630816,0.763622,0.564117,0.453694
56,065_results__data_approach_3__desc_lvl_level_1...,/data/resources/weichel-llama3/work/projects/n...,3,2,True,False,bert-base-uncased,2,0.45,False,...,651.0,27.0,36,0.526164,0.424965,0.717124,0.521005,0.337139,0.526164,0.404593


In [11]:
df_runs.to_csv(os.path.join(BERT_models_path, "trainings_overview.csv"))

In [ ]:
get_data_agg_version = lambda x : x.split("__data_approach_")[1][0]
df_runs["data_agg_approach"] = df_runs["run_name"].apply(get_data_agg_version)

In [ ]:
df_runs["run_name"]

31    037_results__data_approach_3__desc_lvl_level_1...
30    036_results__data_approach_3__desc_lvl_level_1...
6     018_results__data_approach_3__desc_lvl_level_1...
16    019_results__data_approach_3__desc_lvl_level_1...
35    041_results__data_approach_3__desc_lvl_level_1...
32    038_results__data_approach_3__desc_lvl_level_1...
18    024_results__data_approach_3__desc_lvl_level_1...
4     021_results__data_approach_3__desc_lvl_level_1...
38    044_results__data_approach_3__desc_lvl_level_1...
28    034_results__data_approach_1__desc_lvl_level_1...
37    043_results__data_approach_3__desc_lvl_level_1...
21    027_results__data_approach_3__desc_lvl_level_1...
7     016_results__data_approach_1__desc_lvl_level_1...
41    047_results__data_approach_3__desc_lvl_level_1...
15    015_results__data_approach_1__desc_lvl_level_1...
27    033_results__data_approach_1__desc_lvl_level_1...
33    039_results__data_approach_1__desc_lvl_level_1...
39    045_results__data_approach_1__desc_lvl_lev

31       2
30       2
6        2
16       2
35       2
32       2
18       2
4        2
38       2
28       1
37       2
21       2
7        1
41       2
15       1
27       1
33       1
39       1
8        1
2     None
24       2
34       1
19       1
12       1
11       1
44    None
23       1
17       1
29       1
47    None
36       1
46    None
49    None
0     None
1     None
43    None
5        3
45    None
3     None
9        1
48    None
20       2
14       3
40       1
10       1
42       2
13       1
22       1
25       2
26       2
Name: data_path, dtype: object